# Document Translation API Tutorial

This notebook shows how to translate a **whole document** (PDF, Word, Excel, PowerPoint, or HTML) into one or more Indian languages using Sarvam's [Document Translation API](https://docs.sarvam.ai/api/api-guides-tutorials/doc-translation/overview).

Unlike the [Text Translation API](https://docs.sarvam.ai/api/api-guides-tutorials/text-processing/translation), you upload a file and download a **translated file** with layout preserved — document-to-document translation (PDF stays PDF, DOCX stays DOCX).

**Flow:** create job → `PUT` file to signed `upload_url` → `start` → poll `live-status` → `export` each completed target language.

Only translating **plain text** (not a file)? Use the [Translate API tutorial](../translate/Translate_API_Tutorial.ipynb) instead: one request, no upload, no polling.


## 1. Installation


In [ ]:
!pip install -Uqq sarvamai httpx


## 2. Authentication

1. Sign up at the [Sarvam AI Dashboard](https://dashboard.sarvam.ai/) and create an API key.
2. Set `SARVAM_API_KEY` in your environment, or replace the placeholder below.


In [ ]:
import os

SARVAM_API_KEY = os.getenv("SARVAM_API_KEY", "YOUR_SARVAM_API_KEY")
if SARVAM_API_KEY == "YOUR_SARVAM_API_KEY":
    raise ValueError("Set SARVAM_API_KEY before running this notebook.")


## 3. Initialize the client


In [ ]:
from pathlib import Path

import httpx
from sarvamai import SarvamAI
from sarvamai.core.api_error import ApiError

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)


## 4. Create job and upload

Point `document` at a local PDF or Office file. Use the correct `Content-Type` for Office formats (see [supported formats](https://docs.sarvam.ai/api/api-guides-tutorials/doc-translation/overview#supported-file-formats)).


In [ ]:
document = Path("sample.pdf")  # replace with your file

job = client.document_translation.create(
    job_name="sample-pdf",
    source_language_code="en-IN",
    target_language_codes=["hi-IN", "ta-IN"],
    original_filename=document.name,
)

with document.open("rb") as f:
    upload = httpx.put(
        job.upload_url,
        content=f.read(),
        headers={
            "Content-Type": "application/pdf",
            "x-ms-blob-type": "BlockBlob",
        },
        timeout=120.0,
    )
    upload.raise_for_status()

print("job_id:", job.job_id)


## 5. Start the pipeline and poll


In [ ]:
import time

client.document_translation.start(job_id=job.job_id)

while True:
    status = client.document_translation.get_live_status(job_id=job.job_id)
    print("progress:", status.progress_percentage)
    translations = status.translations or []
    if translations and all(t.state == "Completed" for t in translations):
        break
    if status.job_state in ("Failed", "Rejected"):
        raise RuntimeError(status.job_state)
    time.sleep(12)


## 6. Export each target language

Export is **per language**. Call `trigger_export` only after that language's `state` is `Completed`, then poll `get_export_status` for the download URL.


In [ ]:
downloads = {}

for t in status.translations or []:
    if t.state != "Completed":
        continue
    export = client.document_translation.trigger_export(
        job_id=job.job_id,
        lang=t.target_language_code,
    )
    while True:
        export_status = client.document_translation.get_export_status(
            job_id=job.job_id,
            export_id=export.export_id,
        )
        if export_status.export_state == "Completed":
            downloads[t.target_language_code] = export_status.download_url
            break
        if export_status.export_state == "Failed":
            print(f"Export failed for {t.target_language_code}")
            break
        time.sleep(5)

print("Download URLs:", downloads)


## Next steps

- [Poll and export guide](https://docs.sarvam.ai/api/api-guides-tutorials/doc-translation/how-to/poll-and-export)
- [Style guidelines](https://docs.sarvam.ai/api/api-guides-tutorials/doc-translation/how-to/style-guidelines) and [genre selection](https://docs.sarvam.ai/api/api-guides-tutorials/doc-translation/how-to/genre-and-model-tier)
- [FAQs](https://docs.sarvam.ai/api/api-guides-tutorials/doc-translation/faq)
